In [4]:
import math

from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math

from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

In [5]:
class ParametricMethod:
    def __init__(self,filename,p,logTrue=False,distribution=stats.gamma):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.distribution = distribution

    def generateOutput(self):
        self._readArff()
        for v in range(2,70):
            self._distanceMetric(v)
            self._generateArray()
            if self.logTrue:
                self.arr = np.log(self.arr)
            params = self.distribution.fit(self.arr) # fit params for gamma distribution
            posNeg4 = []
            spaceStep4 = np.linspace(0,.99,30) # threshold from 0 to .99, 30 samples
            for e in spaceStep4:
                if len(params) == 3:
                    newArr = self.arr > self.distribution.ppf(e,params[0],loc=params[1], scale=params[2]) # if arr value is outside threshold add to new array
                else:
                    newArr = self.arr > self.distribution.ppf(e,loc=params[0], scale=params[1]) # if arr value is outside threshold add to new array
                posNeg4.append([((self.y[newArr] == 1).sum() / (self.y == 1).sum()), (self.y[newArr] != 1).sum()/ ((self.y != 1).sum())]) # True positive rate, false positive rate

            posNeg4 = np.array(posNeg4)
            arrtest1, arrtest2 = np.split(posNeg4, 2,axis=1) # split the array
            self.tots += [auc(arrtest2, arrtest1)] # return the area under the curve

        self._printResults(self.tots)

    def _distanceMetric(self,n):
        #find the nearestNeighbors
        nn = NearestNeighbors(n_neighbors=n,p=self.p)
        nn.fit(self.X, self.y)
        #return the dist of each and the nearest neighbors
        self.distance, knn = nn.kneighbors(self.X)  # returns N index neighbors including self

    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _generateArray(self):
        self.arr = []
        #returns an array based on the median and max values
        for x in self.distance:  # finds the distance away from that point (index 0)
            self.arr += [np.max(x)]

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        print(max(newarr),newarr.index(max(newarr))+2) #print the max values, the k value, and the array

In [6]:
positively_skewed_distributions = [
    stats.expon,
    stats.chi2,
    stats.gamma,
    stats.weibull_min,
    stats.lognorm,
    stats.invgauss,
    stats.rayleigh,
    stats.wald,
    stats.pareto,
    stats.levy,
    stats.nakagami,
    stats.logistic,
    stats.powerlaw
]

folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]





In [ ]:
for z in folder_structure_1d:
    print(z)
    for i in positively_skewed_distributions:
        #print(i)
        holder = ParametricMethod(z,1,distribution=i)
        holder.generateOutput()

semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff
0.6761145800501458 2
0.6763770930764142 2
